In [1]:
import time
import os
import json
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as transforms
from PIL import Image, ImageDraw, ImageFont

from torchvision.models import mobilenet_v2, MobileNet_V2_Weights, resnet50, vgg16
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign
from torch.cuda.amp import autocast, GradScaler

In [2]:
###########################################
# 1. Dataset e Collate Function
###########################################
class MultiTaskObjectDetectionDataset(Dataset):
    def __init__(self, images_dir, labels_dir, transform=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transform = transform

        # Filtra as imagens que possuem JSON correspondente
        all_ids = [f.split('.')[0] for f in os.listdir(images_dir) if f.endswith('.jpg')]
        self.image_ids = []
        missing = []
        for img_id in all_ids:
            json_path = os.path.join(labels_dir, img_id + '.json')
            if os.path.exists(json_path):
                self.image_ids.append(img_id)
            else:
                missing.append(img_id)
        if missing:
            print("Os seguintes arquivos de imagem não possuem JSON:")
            for m in missing:
                print(m)
        else:
            print("Todas as imagens possuem JSON correspondente.")

        # Carrega os mapeamentos (assumindo que estão na pasta 'helpers')
        with open('helpers/categories.json', 'r') as f:
            self.category_to_label = json.load(f)
        with open('helpers/weather.json', 'r') as f:
            self.weather_to_label = json.load(f)
        with open('helpers/scene.json', 'r') as f:
            self.scene_to_label = json.load(f)
        with open('helpers/timeofday.json', 'r') as f:
            self.timeofday_to_label = json.load(f)

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        try:
            image = Image.open(os.path.join(self.images_dir, image_id + ".jpg")).convert("RGB")
            with open(os.path.join(self.labels_dir, image_id + ".json")) as f:
                data = json.load(f)
        except Exception as e:
            print(f"❌ Erro ao carregar {image_id}: {e}")
            return None

        boxes, labels = [], []
        for obj in data["frames"][0]["objects"]:
            category = obj.get("category")
            if category not in self.category_to_label:
                continue
            bbox = None
            if "box2d" in obj:
                b = obj["box2d"]
                bbox = [b["x1"], b["y1"], b["x2"], b["y2"]]
            elif "poly2d" in obj:
                pts_raw = obj["poly2d"]
                if isinstance(pts_raw[0], dict) and "vertices" in pts_raw[0]:
                    pts = pts_raw[0]["vertices"]
                else:
                    pts = [(p[0], p[1]) for p in pts_raw if isinstance(p, (list, tuple)) and len(p) >= 2]
                if len(pts) >= 2:
                    xs, ys = zip(*pts)
                    bbox = [min(xs), min(ys), max(xs), max(ys)]
            if bbox and bbox[2] > bbox[0] and bbox[3] > bbox[1]:
                boxes.append(bbox)
                labels.append(self.category_to_label[category])

        if not boxes:
            return None

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels}

        attrs = data["attributes"]
        global_attrs = {
            "weather": torch.tensor(self.weather_to_label.get(attrs["weather"], 0)),
            "scene": torch.tensor(self.scene_to_label.get(attrs["scene"], 0)),
            "timeofday": torch.tensor(self.timeofday_to_label.get(attrs["timeofday"], 0)),
        }

        if self.transform:
            image = self.transform(image)

        return image, target, global_attrs
        
def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return ([], [], [])
    images, targets, global_attrs = zip(*batch)
    return list(images), list(targets), list(global_attrs)

In [3]:

###########################################
# 2. Funções de Geração de Modelos
###########################################
def create_mobilenet_model(num_classes):
    mobilenet = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V2)
    backbone = mobilenet.features
    backbone.out_channels = 1280
    anchor_gen = AnchorGenerator(sizes=((32,64,128,256,512),),
                                 aspect_ratios=((0.5,1.0,2.0),))
    roi_pool = MultiScaleRoIAlign(featmap_names=["0"], output_size=7, sampling_ratio=2)
    model = FasterRCNN(backbone, num_classes=num_classes, rpn_anchor_generator=anchor_gen, box_roi_pool=roi_pool)
    return model, backbone, 1280

def create_resnet_model(num_classes):
    backbone = resnet50(pretrained=True)
    modules = list(backbone.children())[:-2]
    backbone = nn.Sequential(*modules)
    backbone.out_channels = 2048
    anchor_gen = AnchorGenerator(sizes=((32,64,128,256,512),),
                                 aspect_ratios=((0.5,1.0,2.0),))
    roi_pool = MultiScaleRoIAlign(featmap_names=["0"], output_size=7, sampling_ratio=2)
    model = FasterRCNN(backbone, num_classes=num_classes, rpn_anchor_generator=anchor_gen, box_roi_pool=roi_pool)
    return model, backbone, 2048

def create_vgg_model(num_classes):
    backbone = vgg16(pretrained=True).features
    backbone.out_channels = 512
    anchor_gen = AnchorGenerator(sizes=((32,64,128,256,512),),
                                 aspect_ratios=((0.5,1.0,2.0),))
    roi_pool = MultiScaleRoIAlign(featmap_names=["0"], output_size=7, sampling_ratio=2)
    model = FasterRCNN(backbone, num_classes=num_classes, rpn_anchor_generator=anchor_gen, box_roi_pool=roi_pool)
    return model, backbone, 512

def create_yolo_model(num_classes):
    class SimpleYOLO(nn.Module):
        def __init__(self, num_classes):
            super(SimpleYOLO, self).__init__()
            self.features = nn.Sequential(
                nn.Conv2d(3, 32, kernel_size=3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(),
                nn.MaxPool2d(2),  # Reduz pela metade
                nn.Conv2d(32, 64, kernel_size=3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(),
                nn.MaxPool2d(2),  # Reduz pela metade
                nn.AdaptiveAvgPool2d((7, 7))  # Garantir saída fixa 7x7
            )

            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(64 * 7 * 7, 256),
                nn.ReLU(),
                nn.Linear(256, num_classes)
            )

            self.box_regressor = nn.Sequential(
                nn.Flatten(),
                nn.Linear(64 * 7 * 7, 256),
                nn.ReLU(),
                nn.Linear(256, 4)
            )

        def forward(self, images, targets=None):
            features = [self.features(img.unsqueeze(0)) for img in images]
            logits = torch.cat([self.classifier(feat) for feat in features], dim=0)
            boxes = torch.cat([self.box_regressor(feat) for feat in features], dim=0)

            if targets is None:
                pred_labels = logits.argmax(dim=1)
                pred_scores = torch.softmax(logits, dim=1).max(dim=1).values
                return [{"boxes": boxes, "labels": pred_labels, "scores": pred_scores}]

            labels = torch.cat([t["labels"] for t in targets], dim=0)
            gt_boxes = torch.cat([t["boxes"] for t in targets], dim=0)

            cls_loss = nn.functional.cross_entropy(logits, labels)
            box_loss = nn.functional.l1_loss(boxes, gt_boxes)

            losses = {
                "classification_loss": cls_loss,
                "bbox_regression_loss": box_loss
            }

            return losses

    model = SimpleYOLO(num_classes)
    return model, None, None

def create_scratch_model(num_classes):
    class SimpleDetector(nn.Module):
        def __init__(self, num_classes):
            super(SimpleDetector, self).__init__()
            self.features = nn.Sequential(
                nn.Conv2d(3, 16, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(16, 32, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.AdaptiveAvgPool2d((7, 7))  # Redimensiona para 7x7 sempre
            )

            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(32 * 7 * 7, 128),
                nn.ReLU(),
                nn.Linear(128, num_classes)
            )

            self.box_regressor = nn.Sequential(
                nn.Flatten(),
                nn.Linear(32 * 7 * 7, 128),
                nn.ReLU(),
                nn.Linear(128, 4)
            )

        def forward(self, images, targets=None):
            features = [self.features(img.unsqueeze(0)) for img in images]
            logits = torch.cat([self.classifier(feat) for feat in features], dim=0)
            boxes = torch.cat([self.box_regressor(feat) for feat in features], dim=0)

            if targets is None:
                pred_labels = logits.argmax(dim=1)
                pred_scores = torch.softmax(logits, dim=1).max(dim=1).values
                return [{"boxes": boxes, "labels": pred_labels, "scores": pred_scores}]

            labels = torch.cat([t["labels"] for t in targets], dim=0)
            gt_boxes = torch.cat([t["boxes"] for t in targets], dim=0)

            cls_loss = nn.functional.cross_entropy(logits, labels)
            box_loss = nn.functional.l1_loss(boxes, gt_boxes)

            losses = {
                "classification_loss": cls_loss,
                "bbox_regression_loss": box_loss
            }

            return losses

    model = SimpleDetector(num_classes)
    return model, None, None

In [4]:
###########################################
# 3. Classe MultiTaskModel (Detecção + Atributos Globais)
###########################################
class MultiTaskModel(nn.Module):
    def __init__(self, detection_model, backbone, num_weather, num_scene, num_time, in_features):
        super(MultiTaskModel, self).__init__()
        self.detection_model = detection_model
        self.backbone = backbone
        self.attr_pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc_weather = nn.Linear(in_features, num_weather)
        self.fc_scene = nn.Linear(in_features, num_scene)
        self.fc_timeofday = nn.Linear(in_features, num_time)
    def forward(self, images, targets=None, global_attrs=None):
        # Em treinamento, retorna losses; em avaliação, retorna as detecções
        if self.training:
            detection_loss = self.detection_model(images, targets)
        else:
            detection_loss = self.detection_model(images)
        imgs_tensor = torch.stack(images)
        feats = self.backbone(imgs_tensor)
        pooled = self.attr_pool(feats)
        pooled = pooled.view(pooled.size(0), -1)
        weather_logits = self.fc_weather(pooled)
        scene_logits = self.fc_scene(pooled)
        timeofday_logits = self.fc_timeofday(pooled)
        if self.training and global_attrs is not None:
            weather_labels = torch.stack([attr["weather"] for attr in global_attrs]).to(images[0].device)
            scene_labels = torch.stack([attr["scene"] for attr in global_attrs]).to(images[0].device)
            timeofday_labels = torch.stack([attr["timeofday"] for attr in global_attrs]).to(images[0].device)
            loss_weather = nn.functional.cross_entropy(weather_logits, weather_labels)
            loss_scene = nn.functional.cross_entropy(scene_logits, scene_labels)
            loss_timeofday = nn.functional.cross_entropy(timeofday_logits, timeofday_labels)
            attr_loss = loss_weather + loss_scene + loss_timeofday
        else:
            attr_loss = 0
        return detection_loss, attr_loss, weather_logits, scene_logits, timeofday_logits

In [ ]:
###########################################
# 4. Pipeline de Treinamento, Avaliação e Inferência
###########################################
# Configurações iniciais
transform = transforms.Compose([transforms.ToTensor()])
splits = {
    "train": ("images/train", "labels/train"),
    "val": ("images/val", "labels/val"),
    "test": ("images/test", "labels/test")
}
datasets, loaders = {}, {}
for split, (img_dir, lbl_dir) in splits.items():
    print(f"📂 Carregando dataset '{split}'...")
    ds = MultiTaskObjectDetectionDataset(img_dir, lbl_dir, transform)
    subset_size = max(1, int(len(ds) * 0.50))  # Usa 50% dos dados
    indices = random.sample(range(len(ds)), subset_size)
    ds_subset = Subset(ds, indices)
    datasets[split] = ds_subset
    loaders[split] = DataLoader(
        ds_subset, batch_size=8, shuffle=(split=="train"),
        collate_fn=collate_fn, num_workers=0)
    print(f"Utilizando {subset_size} de {len(ds)} imagens para o split '{split}' (50%).")

# Carrega os mapeamentos
with open("helpers/categories.json", "r") as f:
    category_to_label = json.load(f)
with open("helpers/weather.json", "r") as f:
    weather_to_label = json.load(f)
with open("helpers/scene.json", "r") as f:
    scene_to_label = json.load(f)
with open("helpers/timeofday.json", "r") as f:
    timeofday_to_label = json.load(f)

num_classes = len(category_to_label) + 1
num_weather = len(weather_to_label)
num_scene = len(scene_to_label)
num_time = len(timeofday_to_label)

# Define o dispositivo (GPU se disponível)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Usando dispositivo: {device}")

# Escolha do modelo: "mobilenet", "resnet", "vgg", "yolo", "scratch"
model_type = "mobilenet"  # Altere conforme necessário
if model_type == "mobilenet":
    detection_model, backbone, in_features = create_mobilenet_model(num_classes)
elif model_type == "resnet":
    detection_model, backbone, in_features = create_resnet_model(num_classes)
elif model_type == "vgg":
    detection_model, backbone, in_features = create_vgg_model(num_classes)
elif model_type == "yolo":
    detection_model, backbone, in_features = create_yolo_model(num_classes)
elif model_type == "scratch":
    detection_model, backbone, in_features = create_scratch_model(num_classes)
else:
    raise ValueError("Modelo inválido.")

# Congelar camadas iniciais
for param in backbone.parameters():
    param.requires_grad = False
    
if backbone is None or in_features is None:
    print("Modelo sem branch de atributos global. Utilizando apenas detecção simples.")
    multi_task_model = detection_model
else:
    multi_task_model = MultiTaskModel(detection_model, backbone, num_weather, num_scene, num_time, in_features=in_features)
multi_task_model = multi_task_model.to(device)

# Congelar camadas iniciais
# Otimizador e AMP
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, multi_task_model.parameters()), lr=1e-4)
scaler = GradScaler()

# Treinamento com Early Stopping (máximo 50 épocas ou sem melhora significativa nas últimas 5 épocas)
max_epochs = 50
early_stop_patience = 5
min_delta = 1e-3
best_loss = float("inf")
patience_counter = 0
loss_history = []

print(f"Modelo tem {sum(p.numel() for p in multi_task_model.parameters())} parâmetros.")
print(f"Número de workers: {loaders['train'].num_workers}")
print(f"Pin memory: {loaders['train'].pin_memory}")
print(f"Batch size: {loaders['train'].batch_size}")
print(f"Dispositivo: {device}")

print("🚀 Iniciando treinamento...")
for epoch in range(max_epochs):
    multi_task_model.train()
    epoch_loss = 0
    epoch_start = time.time()
    total_batches = len(loaders["train"])
    print(f"\n🌀 Epoch {epoch+1}/{max_epochs} - Total Batches: {total_batches}")

    for batch_idx, (images, targets, attrs) in enumerate(loaders["train"], start=1):
        batch_start = time.time()
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        if model_type in ["mobilenet", "resnet", "vgg"]:
            with torch.cuda.amp.autocast():
                det_loss, attr_loss, *_ = multi_task_model(images, targets, attrs)
                loss = sum(det_loss.values()) + attr_loss
        elif model_type in ["yolo", "scratch"]:
            # modelos YOLO e Scratch não têm atributos adicionais,
            det_loss = multi_task_model(images, targets)
            loss = sum(det_loss.values())
        else:
            raise ValueError("Modelo não reconhecido.")
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_time = time.time() - batch_start
        remaining_batches = total_batches - batch_idx
        estimated_remaining_time = remaining_batches * batch_time
        print(f"📦 Epoch {epoch+1}, Batch {batch_idx}/{total_batches} | Loss: {loss.item():.4f} | Time: {batch_time:.2f}s | ⏳ Est. time left: {estimated_remaining_time:.1f}s")

        epoch_loss += loss.item()

    avg_epoch_loss = epoch_loss / total_batches
    epoch_duration = time.time() - epoch_start
    print(f"\n✅ Epoch {epoch+1} complete. Avg Loss: {avg_epoch_loss:.4f}, Duration: {epoch_duration:.2f}s")

    # Criar pasta para logs, se não existir
    os.makedirs("logs", exist_ok=True)
    log_path = f"logs/training_log_{model_type}.json"
    
    # Calcular progresso e ETA
    progress = (epoch + 1) / max_epochs
    estimated_total_time = epoch_duration / progress
    estimated_remaining_time = estimated_total_time - (epoch + 1) * epoch_duration / (epoch + 1)
    
    # Atualiza histórico
    loss_history.append({
        "epoch": epoch + 1,
        "avg_loss": avg_epoch_loss,
        "duration_sec": epoch_duration,
        "best_loss_so_far": best_loss,
        "early_stop_counter": patience_counter,
        "estimated_remaining_time_sec": round(estimated_remaining_time, 1)
    })
    
    # Salvar como JSON
    with open(log_path, "w") as f:
        json.dump(loss_history, f, indent=4)
    print(f"📄 Log de treino salvo em {log_path}")

    # Salvar o modelo completo com o nome baseado no tipo de backbone
    model_filename = f"multi_task_model_{model_type}_{epoch+1}.pth"
    torch.save(multi_task_model, model_filename)
    print(f"💾 Modelo salvo como '{model_filename}'.")

    if best_loss - avg_epoch_loss > min_delta:
        best_loss = avg_epoch_loss
        no_improve_epochs = 0
        torch.save(multi_task_model, f"multi_task_model_{model_type}.pth")
    else:
        no_improve_epochs += 1
        if no_improve_epochs >= patience:
            print("🛑 Early stopping triggered.")
            break

📂 Carregando dataset 'train'...
Todas as imagens possuem JSON correspondente.
Utilizando 3500 de 7000 imagens para o split 'train' (50%).
📂 Carregando dataset 'val'...
Todas as imagens possuem JSON correspondente.
Utilizando 500 de 1000 imagens para o split 'val' (50%).
📂 Carregando dataset 'test'...
Todas as imagens possuem JSON correspondente.
Utilizando 1000 de 2000 imagens para o split 'test' (50%).
🖥️ Usando dispositivo: cuda


C:\Users\ctw02813\AppData\Local\Temp\ipykernel_31512\643864783.py:72: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Modelo tem 82447789 parâmetros.
Número de workers: 0
Pin memory: False
Batch size: 8
Dispositivo: cuda
🚀 Iniciando treinamento...

🌀 Epoch 1/50 - Total Batches: 438


C:\Users\ctw02813\AppData\Local\Temp\ipykernel_31512\643864783.py:103: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


📦 Epoch 1, Batch 1/438 | Loss: 9.1151 | Time: 2.26s | ⏳ Est. time left: 989.0s
📦 Epoch 1, Batch 2/438 | Loss: 7.2980 | Time: 1.77s | ⏳ Est. time left: 772.8s
📦 Epoch 1, Batch 3/438 | Loss: 6.7225 | Time: 1.74s | ⏳ Est. time left: 756.8s
📦 Epoch 1, Batch 4/438 | Loss: 6.7454 | Time: 1.72s | ⏳ Est. time left: 747.0s
📦 Epoch 1, Batch 5/438 | Loss: 6.6790 | Time: 1.46s | ⏳ Est. time left: 631.2s
📦 Epoch 1, Batch 6/438 | Loss: 6.6862 | Time: 1.04s | ⏳ Est. time left: 451.4s


In [ ]:
###########################################
# 5. Avaliação
###########################################
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

def evaluate(model, loader, device, iou_threshold=0.5, confidence_threshold=0.6):
    model.eval()
    total = 0
    correct_weather, correct_scene, correct_time = 0, 0, 0
    total_gt = 0      
    correct_category = 0  
    detections_count = 0  
    with torch.no_grad():
        for images, targets, attrs in loader:
            if not images:
                continue
            images = [img.to(device) for img in images]
            # Predição dos atributos globais
            _, _, w_logits, s_logits, t_logits = model(images)
            w_preds = torch.argmax(w_logits, dim=1).cpu()
            s_preds = torch.argmax(s_logits, dim=1).cpu()
            t_preds = torch.argmax(t_logits, dim=1).cpu()
            w_true = torch.stack([a["weather"] for a in attrs])
            s_true = torch.stack([a["scene"] for a in attrs])
            t_true = torch.stack([a["timeofday"] for a in attrs])
            correct_weather += (w_preds == w_true).sum().item()
            correct_scene += (s_preds == s_true).sum().item()
            correct_time += (t_preds == t_true).sum().item()
            total += len(images)
            # Avaliação de detecções
            detections_batch = model.detection_model(images)
            for i, detection in enumerate(detections_batch):
                gt_boxes = targets[i]["boxes"].cpu().numpy()
                gt_labels = targets[i]["labels"].cpu().numpy()
                total_gt += len(gt_boxes)
                pred_boxes = detection["boxes"].cpu().numpy()
                pred_labels = detection["labels"].cpu().numpy()
                pred_scores = detection["scores"].cpu().numpy()
                keep = pred_scores >= confidence_threshold
                pred_boxes = pred_boxes[keep]
                pred_labels = pred_labels[keep]
                for j, gt_box in enumerate(gt_boxes):
                    best_iou = 0
                    best_idx = -1
                    for k, pred_box in enumerate(pred_boxes):
                        iou = compute_iou(gt_box, pred_box)
                        if iou > best_iou:
                            best_iou = iou
                            best_idx = k
                    if best_iou >= iou_threshold:
                        detections_count += 1
                        if pred_labels[best_idx] == gt_labels[j]:
                            correct_category += 1
    weather_acc = correct_weather / total if total > 0 else 0
    scene_acc = correct_scene / total if total > 0 else 0
    time_acc = correct_time / total if total > 0 else 0
    detection_cat_acc = correct_category / total_gt if total_gt > 0 else 0
    avg_detections = detections_count / total if total > 0 else 0
    print(f"📊 Avaliação em {total} imagens:")
    print(f"🌤️ Acurácia Weather: {weather_acc:.2%}")
    print(f"🌆 Acurácia Scene:   {scene_acc:.2%}")
    print(f"🌙 Acurácia Time:    {time_acc:.2%}")
    print(f"📦 Média de detecções por imagem (matched): {avg_detections:.2f}")
    print(f"🎯 Acurácia de categorias: {detection_cat_acc:.2%}")
    return weather_acc, scene_acc, time_acc, avg_detections, detection_cat_acc

print("Iniciando avaliação...")
evaluate(multi_task_model, loaders["val"], device, iou_threshold=0.5, confidence_threshold=0.6)

In [ ]:
###########################################
# 6. Inferência em múltiplas imagens do conjunto de teste
###########################################
def invert_mapping(mapping):
    return {v: k for k, v in mapping.items()}

def infer_and_annotate_image(image_path, model, device, transform,
                             category_to_label, weather_to_label, 
                             scene_to_label, timeofday_to_label,
                             confidence_threshold=0.5):
    inv_category = invert_mapping(category_to_label)
    inv_weather = invert_mapping(weather_to_label)
    inv_scene = invert_mapping(scene_to_label)
    inv_timeofday = invert_mapping(timeofday_to_label)
    orig_image = Image.open(image_path).convert("RGB")
    input_image = transform(orig_image).to(device).unsqueeze(0)
    model.eval()
    with torch.no_grad():
        detections = model.detection_model([input_image.squeeze(0)])
        feats = model.backbone(input_image)
        pooled = model.attr_pool(feats)
        pooled = pooled.view(pooled.size(0), -1)
        weather_logits = model.fc_weather(pooled)
        scene_logits = model.fc_scene(pooled)
        timeofday_logits = model.fc_timeofday(pooled)
        weather_pred = int(torch.argmax(weather_logits, dim=1).item())
        scene_pred = int(torch.argmax(scene_logits, dim=1).item())
        timeofday_pred = int(torch.argmax(timeofday_logits, dim=1).item())
        global_attributes = {
            "weather": inv_weather.get(weather_pred, str(weather_pred)),
            "scene": inv_scene.get(scene_pred, str(scene_pred)),
            "timeofday": inv_timeofday.get(timeofday_pred, str(timeofday_pred))
        }
        detection = detections[0]
        boxes = detection["boxes"].cpu().numpy().tolist()
        labels = detection["labels"].cpu().numpy().tolist()
        scores = detection["scores"].cpu().numpy().tolist()
        detection_list = []
        for bbox, label, score in zip(boxes, labels, scores):
            if score < confidence_threshold:
                continue
            detection_list.append({
                "category": inv_category.get(label, str(label)),
                "score": score,
                "box": bbox
            })
    draw = ImageDraw.Draw(orig_image)
    try:
        font = ImageFont.truetype("arial.ttf", 15)
    except Exception:
        font = ImageFont.load_default()
    for det in detection_list:
        bbox = det["box"]
        text = f"{det['category']}: {det['score']:.2f}"
        draw.rectangle(bbox, outline="red", width=2)
        draw.text((bbox[0], bbox[1]-10), text, fill="red", font=font)
    attr_text = f"Weather: {global_attributes['weather']}, Scene: {global_attributes['scene']}, Time: {global_attributes['timeofday']}"
    draw.text((10, 10), attr_text, fill="blue", font=font)
    return orig_image, {"global_attributes": global_attributes, "detections": detection_list}

def infer_on_test_folder(test_folder, output_folder, model, device, transform,
                         category_to_label, weather_to_label, scene_to_label, timeofday_to_label,
                         confidence_threshold=0.5):
    os.makedirs(output_folder, exist_ok=True)
    results = {}
    for filename in os.listdir(test_folder):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_path = os.path.join(test_folder, filename)
            annotated_img, output = infer_and_annotate_image(image_path, model, device, transform,
                                                             category_to_label, weather_to_label, scene_to_label, timeofday_to_label,
                                                             confidence_threshold)
            base_name = os.path.splitext(filename)[0]
            annotated_image_path = os.path.join(output_folder, f"{base_name}_annotated.jpg")
            json_output_path = os.path.join(output_folder, f"{base_name}.json")
            annotated_img.save(annotated_image_path)
            with open(json_output_path, "w") as f:
                json.dump(output, f, indent=2)
            print(f"Processado: {filename}")
            results[filename] = output
    return results

print("Iniciando inferência no conjunto de teste...")
test_folder = "images/test"
output_folder = f"{model_type}_outputs"
inference_results = infer_on_test_folder(test_folder, output_folder, multi_task_model, device, transform,
                                         category_to_label, weather_to_label, scene_to_label, timeofday_to_label,
                                         confidence_threshold=0.5)
print("Inferência concluída. Resultados:")
print(json.dumps(inference_results, indent=2))